In [1]:
import os
import time
import io

import requests
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [2]:
def get_chrome_driver():
    try:
        return webdriver.Chrome()
    except Exception as e:
        print(f"Selenium Manager failed ({e}); falling back to webdriver-manager...")
        from webdriver_manager.chrome import ChromeDriverManager
        service = Service(ChromeDriverManager().install())
        return webdriver.Chrome(service=service)

In [3]:
def find_with_fallback(driver, selectors, timeout=10, find_all=False, label=""):
    for selector in selectors:
        try:
            condition = (
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
                if find_all else
                EC.presence_of_element_located((By.CSS_SELECTOR, selector))
            )
            result = WebDriverWait(driver, timeout).until(condition)
            print(f"  [{label}] matched selector: {selector!r}")
            return result
        except TimeoutException:
            continue

    raise TimeoutException(
        f"[{label}] None of these selectors matched: {selectors} "
        f"— the page markup has likely changed; inspect it and add the "
        f"current selector to this list."
    )

In [4]:
def page_is_blocked(driver):
    page_source_lower = driver.page_source.lower()
    current_url_lower = driver.current_url.lower()

    signals = [
        "unusual traffic" in page_source_lower,
        "g-recaptcha" in page_source_lower,
        "/sorry/" in current_url_lower,
        "captcha" in current_url_lower,
    ]

    return any(signals)


def wait_for_manual_solve(driver, context_label, max_prompts=3):
    """Pause and let the user solve a CAPTCHA themselves in the visible
    browser window. Returns True once the page is no longer blocked,
    False if the user gives up or it's still blocked after max_prompts
    attempts.
    """
    for attempt in range(1, max_prompts + 1):
        print(f"\n'{context_label}': CAPTCHA / bot-check page detected.")
        print("-> Switch to the Chrome window Selenium opened (check your Dock).")
        print("-> Solve the challenge there yourself, and wait for the real page to load.")
        answer = input(f"Type 'y' + Enter once solved, or 's' + Enter to skip this search "
                        f"(attempt {attempt}/{max_prompts}): ").strip().lower()

        if answer == 's':
            print(f"'{context_label}': skipping at your request.")
            return False

        if not page_is_blocked(driver):
            print(f"'{context_label}': looks unblocked now — continuing.")
            return True

        print(f"'{context_label}': still looks blocked. ", end="")

    print(f"'{context_label}': giving up after {max_prompts} attempts — skipping this search.")
    return False

In [5]:
MAX_STALLED_ATTEMPTS = 3
MAX_SECONDS_PER_SEARCH = 60

THUMBNAIL_SELECTORS = ["img.Q4LuWd", "img.YQ4gaf", "g-img img", "img[data-src]"]
FULL_IMAGE_SELECTORS = ["img.n3VNCb", "img.sFlh5c", "img.iPVvYb", "img.r48jcc"]
LOAD_MORE_SELECTORS = [".mye4qd", "input.mye4qd"]


def image_urls(string, max_links, driver):

    def scroll_to_end(driver):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

    search_url = "https://www.google.com/search?safe=off&site=&tbm=isch&source=hp&q={q}&oq={q}&gs_l=img"
    driver.get(search_url.format(q=string))

    if page_is_blocked(driver):
        if not wait_for_manual_solve(driver, string):
            return set()
        # user solved it manually — give the real results page a moment to load
        time.sleep(2)

    image_urls = set()
    image_count = 0
    results_start = 0
    stalled_count = 0
    last_image_count = -1
    start_time = time.time()

    while image_count < max_links:
        if time.time() - start_time > MAX_SECONDS_PER_SEARCH:
            print(f"'{string}': hit the {MAX_SECONDS_PER_SEARCH}s time budget. "
                  f"Returning {image_count} image(s) found so far.")
            return image_urls

        scroll_to_end(driver)

        try:
            thumbnail_results = find_with_fallback(
                driver, THUMBNAIL_SELECTORS, find_all=True, label="thumbnails")
        except TimeoutException as e:
            print(e)
            return image_urls

        number_results = len(thumbnail_results)
        print(f"Found: {number_results} search results. Extracting links from {results_start}:{number_results}")

        for img in thumbnail_results[results_start:number_results]:
            try:
                img.click()
                time.sleep(1)
            except Exception:
                continue

            try:
                actual_images = find_with_fallback(
                    driver, FULL_IMAGE_SELECTORS, timeout=3, find_all=True, label="full image")
            except TimeoutException:
                continue

            for actual_image in actual_images:
                src = actual_image.get_attribute('src')
                if src and 'http' in src:
                    image_urls.add(src)

            image_count = len(image_urls)

            if image_count >= max_links:
                print(f"Found: {image_count} image links")
                break
        else:
            if image_count == last_image_count:
                stalled_count += 1
            else:
                stalled_count = 0
            last_image_count = image_count

            if stalled_count >= MAX_STALLED_ATTEMPTS:
                print(f"'{string}': no new images found after {MAX_STALLED_ATTEMPTS} attempts. "
                      f"Returning {image_count} image(s) found so far.")
                return image_urls

            print("Found:", image_count, "looking for more image links ...")
            time.sleep(5)

            try:
                load_more_button = find_with_fallback(
                    driver, LOAD_MORE_SELECTORS, timeout=3, find_all=True, label="load more")
                if load_more_button:
                    driver.execute_script(f"document.querySelector('{LOAD_MORE_SELECTORS[0]}').click();")
            except TimeoutException:
                pass

            results_start = len(thumbnail_results)
            continue

        results_start = len(thumbnail_results)

    return image_urls

In [6]:
def save_images(folder_path, file_name, url):
    try:
        image_content = requests.get(url, timeout=10).content
    except Exception as e:
        print(f"ERROR - COULD NOT DOWNLOAD {url} - {e}")
        return

    try:
        image_file = io.BytesIO(image_content)
        image = Image.open(image_file).convert('RGB')

        file_path = os.path.join(folder_path, file_name)

        with open(file_path, 'wb') as f:
            image.save(f, "JPEG", quality=85)
        print(f"SAVED - {url} - AT: {file_path}")
    except Exception as e:
        print(f"ERROR - COULD NOT SAVE {url} - {e}")

In [7]:
SEARCH_BOX_SELECTORS = ["input.gLFyf", "textarea.gLFyf", "textarea[name='q']", "input[name='q']"]

search_names = ["bus", "cars"]
images_path = './img/'

driver = get_chrome_driver()

try:
    for name in search_names:
        path = os.path.join(images_path, name)
        if not os.path.isdir(path):
            os.makedirs(path)

        driver.get('https://google.com')

        if page_is_blocked(driver):
            if not wait_for_manual_solve(driver, name):
                continue
            time.sleep(2)

        try:
            search_box = find_with_fallback(driver, SEARCH_BOX_SELECTORS, label="search box")
        except TimeoutException as e:
            print(e)
            continue

        search_box.send_keys(name)
        links = image_urls(name, 5, driver)

        for i, link in enumerate(links):
            file_name = f"{i:06}.jpg"
            save_images(path, file_name, link)
finally:
    driver.quit()

print("Done.")

  [search box] matched selector: 'textarea.gLFyf'

'bus': CAPTCHA / bot-check page detected.
-> Switch to the Chrome window Selenium opened (check your Dock).
-> Solve the challenge there yourself, and wait for the real page to load.
'bus': looks unblocked now — continuing.
  [thumbnails] matched selector: 'img.YQ4gaf'
Found: 257 search results. Extracting links from 0:257


KeyboardInterrupt: 